# PD WOE and Information Value

WOE and IV are calculated on development data only. They are screening diagnostics for binary PD targets and are not used for LGD or EAD.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"
CFG = yaml.safe_load((ROOT / "config" / "project.yaml").read_text())

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

WOE = log(distribution of non-defaults / distribution of defaults). IV is the sum of the distribution difference multiplied by WOE.

In [2]:
woe = query('select * from pd_woe_iv')
woe.head()

,variable,bin,count,bad,good,distribution_good,distribution_bad,woe,iv_component,default_rate,variable_iv,model
0,classic_fico,"(484.999, 684.0]",89376,"1,818.0000","87,558.0000",0.1013,0.2643,-0.9587,0.1562,0.0203,0.5399,12-month logistic PD
1,classic_fico,"(684.0, 708.0]",88056,"1,355.0000","86,701.0000",0.1003,0.1970,-0.6747,0.0652,0.0154,0.5399,12-month logistic PD
2,classic_fico,"(708.0, 728.0]",87123,847.0000,"86,276.0000",0.0998,0.1232,-0.2099,0.0049,0.0097,0.5399,12-month logistic PD
3,classic_fico,"(728.0, 746.0]",84393,668.0000,"83,725.0000",0.0969,0.0972,-0.0027,0.0000,0.0079,0.5399,12-month logistic PD
4,classic_fico,"(746.0, 761.0]",89073,734.0000,"88,339.0000",0.1022,0.1067,-0.0432,0.0002,0.0082,0.5399,12-month logistic PD


In [3]:
sample = query("select classic_fico, target, observation_weight from pd_development_sample where target_name='12_month_pd' and split='development'")
sample['fico_band'] = pd.qcut(sample['classic_fico'], 10, duplicates='drop')
groups = sample.groupby('fico_band', observed=False).apply(
    lambda x: pd.Series({'defaults': (x.target*x.observation_weight).sum(),
                         'non_defaults': ((1-x.target)*x.observation_weight).sum()}),
    include_groups=False).reset_index()
epsilon = 0.5
groups['default_share'] = (groups.defaults+epsilon)/(groups.defaults.sum()+epsilon*len(groups))
groups['non_default_share'] = (groups.non_defaults+epsilon)/(groups.non_defaults.sum()+epsilon*len(groups))
groups['woe'] = np.log(groups.non_default_share/groups.default_share)
groups['iv_component'] = (groups.non_default_share-groups.default_share)*groups.woe
groups[['fico_band','defaults','non_defaults','woe','iv_component']]

,fico_band,defaults,non_defaults,woe,iv_component
0,"(484.999, 680.0]","1,621.0000","76,434.8008",-0.9799,0.1442
1,"(680.0, 703.0]","1,294.0000","79,652.0499",-0.7134,0.0685
2,"(703.0, 724.0]",889.0000,"84,452.7888",-0.2797,0.0088
3,"(724.0, 743.0]",720.0000,"86,639.5128",-0.0434,0.0002
4,"(743.0, 758.0]",733.0000,"84,452.7888",-0.0868,0.0008
5,"(758.0, 771.0]",453.0000,"91,012.9608",0.4688,0.0185
6,"(771.0, 782.0]",480.0000,"88,034.4919",0.3777,0.0121
7,"(782.0, 793.0]",237.0000,"94,418.7206",1.1524,0.0861
8,"(793.0, 802.0]",319.0000,"87,607.2010",0.7809,0.0429
9,"(802.0, 829.0]",130.0000,"91,427.6843",1.7190,0.1493


In [4]:
groups.iv_component.sum()

np.float64(0.5313710864617598)

In [5]:
variable_iv = (woe.groupby(['model','variable'], as_index=False)['variable_iv'].max()
.sort_values(['model','variable_iv'], ascending=[True,False]))
variable_iv.groupby('model').head(12)

,model,variable,variable_iv
2,12-month logistic PD,current_dpd,1.2580
8,12-month logistic PD,max_dpd_12m,0.8732
10,12-month logistic PD,max_dpd_6m,0.8265
9,12-month logistic PD,max_dpd_3m,0.7930
16,12-month logistic PD,unemployment_rate,0.6418
0,12-month logistic PD,classic_fico,0.5399
5,12-month logistic PD,gdp_growth_yoy,0.3835
3,12-month logistic PD,current_interest_rate,0.2291
6,12-month logistic PD,hpi_growth_yoy,0.2228
13,12-month logistic PD,original_dti,0.2055


In [6]:
selected = variable_iv.iloc[0]['variable']
woe[woe['variable'].eq(selected)][['model','variable','bin','count','default_rate','woe','iv_component','variable_iv']]

,model,variable,bin,count,default_rate,woe,iv_component,variable_iv
42,12-month logistic PD,current_dpd,0.0,865084,0.0058,0.3056,0.0801,1.2580
43,12-month logistic PD,current_dpd,30.0,4802,0.2170,-3.5505,0.5227,1.2580
44,12-month logistic PD,current_dpd,60.0,1123,0.7053,-5.7050,0.6552,1.2580
158,discrete-time monthly hazard,current_dpd,0.0,912001,0.0000,5.1135,5.0513,12.2836
159,discrete-time monthly hazard,current_dpd,30.0,5094,0.0008,-0.3261,0.0007,12.2836
160,discrete-time monthly hazard,current_dpd,60.0,1166,0.4949,-7.3366,7.2316,12.2836


IV is used with economic judgement, missing-value review, stability and correlation checks. It is not an automatic variable-selection rule.